# — GroupBy, Aggregation & Merging
**NAVTTC · Data Analytics Using Python · 02 April 2026**

---

**Dataset:** `ecommerce_orders.csv` — 515 Pakistani e-commerce orders, Jan 2023–Jun 2024  
**Columns:** `order_id`, `customer_name`, `city`, `category`, `product`, `quantity`, `unit_price`, `payment_method`, `order_status`, `order_date`, `rating`

| Section | Topic |
|---------|-------|
| 0 | Setup, load, inspect |
| 1 | Fix the data |
| 2 | groupby — single column |
| 3 | agg() — multiple stats per group |
| 4 | Multi-column groupby |
| 5 | pd.merge() — join two tables |
| 6 | pd.concat() — stack multiple files |
| 7 | **Independent practice — 10 questions** |

> Every comment explains *why*. Read them.

---
## Section 0 — Setup & Inspection

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', 15)
pd.set_option('display.float_format', '{:,.1f}'.format)

df = pd.read_csv('ecommerce_orders.csv')
print(f'Rows: {df.shape[0]}   Columns: {df.shape[1]}')
df.head()

In [ ]:
# info() shows dtypes AND non-null counts in one call
# Non-null < total rows = that column has missing values
df.info()

In [ ]:
# Find the three problems that always exist in real data:
# 1. Missing values
# 2. Wrong data types
# 3. Inconsistent string formatting

print('Missing values per column:')
print(df.isnull().sum())
print()

print('unit_price dtype:', df['unit_price'].dtype)
print('Sample values:', df['unit_price'].sample(12, random_state=1).tolist())
print()

print('City values (raw):')
print(sorted(df['city'].unique()))

---
## Section 1 — Fix the Data

Problems:
- `city` — mixed case, trailing spaces (`'karachi'`, `'LAHORE'`, `' Rawalpindi'`)
- `unit_price` — stored as string, some rows have `Rs.` prefix or extra spaces
- `quantity` — 3 rows have negative values (data entry error)
- `customer_name` — 8 NaN rows
- 10 exact duplicate rows

In [ ]:
# Fix city: strip whitespace → title case
df['city'] = df['city'].str.strip().str.title()

# Fix customer_name: strip → title → fill NaN
df['customer_name'] = df['customer_name'].str.strip().str.title().fillna('Unknown')

# Fix unit_price: remove 'Rs.' → strip → convert to number
# errors='coerce' turns anything that can't convert into NaN (not a crash)
df['unit_price'] = pd.to_numeric(
    df['unit_price'].astype(str).str.replace('Rs.', '', regex=False).str.strip(),
    errors='coerce'
)

# Fix order_date: convert string to datetime
df['order_date'] = pd.to_datetime(df['order_date'], errors='coerce')

# Remove invalid rows
before = len(df)
df = df.dropna(subset=['unit_price'])  # can't calculate revenue without a price
df = df[df['quantity'] > 0]           # negative quantity = data entry error
df = df.drop_duplicates()             # remove identical duplicate rows

print(f'Rows removed: {before - len(df)}   Clean rows: {len(df)}')

# Calculate total_revenue — every analysis today uses this
df['total_revenue'] = df['quantity'] * df['unit_price']

print(f'Total revenue: PKR {df["total_revenue"].sum():,.0f}')

In [ ]:
# Confirm everything is clean before moving on
print('Dtypes after fixing:')
print(df[['city', 'unit_price', 'order_date', 'total_revenue']].dtypes)
print()
print('Missing values after fixing:')
print(df.isnull().sum())
print()
print('Cities:')
print(sorted(df['city'].unique()))

---
## Section 2 — groupby() Basics

**Pattern:** `df.groupby('column')['value_col'].function()`

Three steps internally: **Split** rows by group → **Apply** function → **Combine** into result.

In [ ]:
# Revenue per city — sorted highest to lowest
city_revenue = (
    df.groupby('city')['total_revenue']
    .sum()
    .sort_values(ascending=False)
)
print(city_revenue)

In [ ]:
# The result is a Series with city as index
# reset_index() converts it back to a regular DataFrame with numbered rows
city_df = city_revenue.reset_index()
print(type(city_revenue))  # Series
print(type(city_df))       # DataFrame
print(city_df.head())

In [ ]:
# Visualise immediately
fig, ax = plt.subplots(figsize=(10, 4))
colors = ['#F59E0B'] + ['#0D9488'] * (len(city_revenue) - 1)
ax.bar(city_revenue.index, city_revenue.values / 1000, color=colors, edgecolor='white')
ax.set_title('Total Revenue by City  (PKR thousands)', fontweight='bold')
ax.set_ylabel("Revenue (PKR '000)")
ax.tick_params(axis='x', rotation=35)
ax.grid(axis='y', alpha=0.3)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig('city_revenue.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# All common single-function groupby calls
g = df.groupby('category')['total_revenue']

print('sum()     total revenue per category:')
print(g.sum().sort_values(ascending=False))
print()
print('mean()    average order value per category:')
print(g.mean().sort_values(ascending=False).round(0))
print()
print('count()   number of orders per category:')
print(g.count().sort_values(ascending=False))
print()
print('max()     largest single order per category:')
print(g.max().sort_values(ascending=False))

In [ ]:
# nunique() — how many unique products per category
print('Unique products per category:')
print(df.groupby('category')['product'].nunique().sort_values(ascending=False))

In [ ]:
# filter() — keep only groups that meet a condition
# Returns rows (not a summary) from groups that pass the test
cities_with_30plus = df.groupby('city').filter(lambda x: len(x) >= 30)

print(f'All cities:              {df["city"].nunique()}')
print(f'Cities with 30+ orders:  {cities_with_30plus["city"].nunique()}')
print(f'Rows kept:               {len(cities_with_30plus)}')

In [ ]:
# Monthly revenue trend using datetime
df['year_month'] = df['order_date'].dt.to_period('M')  # '2023-01', '2023-02', ...

monthly = (
    df.groupby('year_month')['total_revenue']
    .sum()
    .sort_index()
)

fig, ax = plt.subplots(figsize=(12, 4))
months_str = [str(m) for m in monthly.index]
ax.fill_between(months_str, monthly.values / 1000, alpha=0.25, color='#0D9488')
ax.plot(months_str, monthly.values / 1000, color='#0D9488', linewidth=2, marker='o', markersize=4)
ax.set_title('Monthly Revenue Trend  (PKR thousands)', fontweight='bold')
ax.set_ylabel("Revenue (PKR '000)")
ax.tick_params(axis='x', rotation=45)
ax.grid(axis='y', alpha=0.3)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig('monthly_trend.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Section 3 — agg() — Multiple Stats in One Call

**Syntax:** `groupby('col').agg(new_col=('source_col', 'function'))`

One groupby — as many output columns as you need.

In [ ]:
# Complete category performance report
category_report = df.groupby('category').agg(
    total_revenue    = ('total_revenue', 'sum'),
    order_count      = ('order_id',      'count'),
    avg_order_value  = ('total_revenue', 'mean'),
    avg_rating       = ('rating',        'mean'),    # NaN is ignored automatically
    unique_products  = ('product',       'nunique'),
    max_single_order = ('total_revenue', 'max'),
).sort_values('total_revenue', ascending=False).round(0)

# Add revenue share after the agg — regular column operation
category_report['revenue_share_%'] = (
    category_report['total_revenue'] / category_report['total_revenue'].sum() * 100
).round(1)

print(category_report)

In [ ]:
# Two charts side by side
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

colors = ['#0D9488' if i > 0 else '#F59E0B' for i in range(len(category_report))]
ax1.barh(category_report.index, category_report['total_revenue'] / 1000, color=colors, edgecolor='white')
ax1.set_title('Total Revenue by Category  (PKR thousands)', fontweight='bold')
ax1.set_xlabel("Revenue (PKR '000)")
ax1.grid(axis='x', alpha=0.3)
ax1.spines[['top', 'right']].set_visible(False)

ax2.barh(category_report.index, category_report['avg_rating'], color='#7C3AED', edgecolor='white')
ax2.axvline(x=category_report['avg_rating'].mean(), color='#F59E0B',
            linestyle='--', label='Overall average')
ax2.set_title('Average Rating by Category', fontweight='bold')
ax2.set_xlabel('Average Rating (out of 5)')
ax2.set_xlim(0, 5.5)
ax2.legend(fontsize=9)
ax2.grid(axis='x', alpha=0.3)
ax2.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.savefig('category_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Payment method — mean vs median matters here
# Large gap between them = a few very large orders skewing the mean
payment_report = df.groupby('payment_method').agg(
    order_count   = ('order_id',      'count'),
    mean_order    = ('total_revenue', 'mean'),
    median_order  = ('total_revenue', 'median'),
    std_order     = ('total_revenue', 'std'),
).sort_values('mean_order', ascending=False).round(0)

payment_report['mean_median_gap'] = (
    payment_report['mean_order'] - payment_report['median_order']
).round(0)

print('Payment method analysis:')
print(payment_report)

In [ ]:
# Order status breakdown — what % of orders are cancelled?
status_summary = df.groupby('order_status').agg(
    order_count   = ('order_id',      'count'),
    total_revenue = ('total_revenue', 'sum'),
    avg_revenue   = ('total_revenue', 'mean'),
).sort_values('order_count', ascending=False)

status_summary['pct_of_orders'] = (
    status_summary['order_count'] / status_summary['order_count'].sum() * 100
).round(1)

print(status_summary)

In [ ]:
# Top 10 customers — including their favourite category
top10 = (
    df.groupby('customer_name').agg(
        total_spend   = ('total_revenue', 'sum'),
        order_count   = ('order_id',      'count'),
        avg_spend     = ('total_revenue', 'mean'),
        fav_category  = ('category',      lambda x: x.value_counts().index[0]),
        last_city     = ('city',          'last'),
    )
    .sort_values('total_spend', ascending=False)
    .head(10)
)
print(top10)

---
## Section 4 — Multi-Column GroupBy

`df.groupby(['col1', 'col2'])` — groups by every unique combination of the two columns.

In [ ]:
# City × Category — which city buys the most of each category?
# unstack() pivots the inner group level to columns
city_cat = (
    df.groupby(['city', 'category'])['total_revenue']
    .sum()
    .unstack(fill_value=0)   # fill_value=0 → no NaN where combo didn't occur
)
print(city_cat)

In [ ]:
# Heatmap — the right chart when you have two categorical dimensions
fig, ax = plt.subplots(figsize=(13, 6))
sns.heatmap(
    city_cat / 1000,
    annot=True, fmt='.0f',
    cmap='YlOrRd',
    linewidths=0.5,
    ax=ax
)
ax.set_title("Revenue by City x Category  (PKR thousands)", fontsize=13, fontweight='bold', pad=12)
ax.set_xlabel('Product Category')
ax.set_ylabel('City')
ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.savefig('heatmap_city_category.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Which city is best for each category?
print('Best city per category:')
for cat in city_cat.columns:
    best = city_cat[cat].idxmax()
    rev  = city_cat[cat].max()
    print(f'  {cat:<22} -> {best:<15}  PKR {rev:,.0f}')

In [ ]:
# Category × Order status — which categories get cancelled the most?
cat_status = (
    df.groupby(['category', 'order_status'])['order_id']
    .count()
    .unstack(fill_value=0)
)

cat_status['cancellation_rate_%'] = (
    cat_status.get('Cancelled', 0) / cat_status.sum(axis=1) * 100
).round(1)

print(cat_status[['Delivered', 'Cancelled', 'cancellation_rate_%']]
      .sort_values('cancellation_rate_%', ascending=False))

In [ ]:
# Combine multiple Series into one DataFrame using concat axis=1
rev_by_city   = df.groupby('city')['total_revenue'].sum().rename('revenue')
cnt_by_city   = df.groupby('city')['order_id'].count().rename('orders')
rating_by_city = df.groupby('city')['rating'].mean().rename('avg_rating').round(2)

city_summary = pd.concat([rev_by_city, cnt_by_city, rating_by_city], axis=1)
city_summary['revenue_per_order'] = (city_summary['revenue'] / city_summary['orders']).round(0)
city_summary = city_summary.sort_values('revenue', ascending=False)

print(city_summary)

---
## Section 5 — pd.merge() — Join Two Tables

```python
result = pd.merge(left_df, right_df, on='shared_key', how='left')
```

| `how=` | Keeps |
|--------|-------|
| `'inner'` | Only rows matched in BOTH tables |
| `'left'` | ALL rows from left, matched from right (NaN where no match) |
| `'right'` | ALL rows from right, matched from left |
| `'outer'` | ALL rows from both tables |

In [ ]:
# Build a customers reference table
np.random.seed(42)
unique_names = df['customer_name'].dropna().unique()

customers = pd.DataFrame({
    'customer_name':  unique_names,
    'tier':           np.random.choice(['Bronze', 'Silver', 'Gold', 'Platinum'],
                                       size=len(unique_names), p=[0.4, 0.35, 0.18, 0.07]),
    'registered_yr':  np.random.choice([2020, 2021, 2022, 2023], size=len(unique_names)),
    'discount_pct':   np.random.choice([0, 5, 10, 15], size=len(unique_names)),
    'account_manager':np.random.choice(['Bilal Ahmed', 'Sara Khan', 'Omar Qureshi'],
                                        size=len(unique_names)),
})

print(f'Customers table: {customers.shape}')
print(customers.head(6))

In [ ]:
# LEFT merge: add customer details to every order
# how='left' keeps ALL orders — even if customer has no matching row in customers table
df_full = pd.merge(
    df,
    customers,
    on='customer_name',
    how='left'
)

print(f'Orders before merge: {len(df)}')
print(f'Orders after merge:  {len(df_full)}   (left merge never drops rows)')
print(f'New columns: {[c for c in df_full.columns if c not in df.columns]}')

In [ ]:
# Now analyse using merged data — which tier drives the most revenue?
tier_report = df_full.groupby('tier').agg(
    total_revenue = ('total_revenue', 'sum'),
    order_count   = ('order_id',      'count'),
    avg_order     = ('total_revenue', 'mean'),
).sort_values('total_revenue', ascending=False)

tier_report['revenue_share_%'] = (
    tier_report['total_revenue'] / tier_report['total_revenue'].sum() * 100
).round(1)

print(tier_report)

In [ ]:
# INNER vs LEFT — understand what you lose
df_inner = pd.merge(df, customers, on='customer_name', how='inner')
df_left  = pd.merge(df, customers, on='customer_name', how='left')

print(f'INNER merge: {len(df_inner)} rows  (dropped unmatched orders)')
print(f'LEFT merge:  {len(df_left)} rows   (kept all orders)')
print(f'Lost orders: {len(df_left) - len(df_inner)}')

In [ ]:
# Merging on different column names in each table
# Use left_on= and right_on= when the key has a different name in each table

products_ref = pd.DataFrame({
    'product_name':  df['product'].unique(),
    'cost_price':    np.random.randint(500, 50000, size=df['product'].nunique()),
    'supplier':      np.random.choice(['LocalSupply PK', 'ImportDirect', 'WholesaleMart'],
                                       size=df['product'].nunique()),
})

# Left table column: 'product'   Right table column: 'product_name'
df_with_cost = pd.merge(
    df,
    products_ref,
    left_on='product',
    right_on='product_name',
    how='left'
)

df_with_cost['profit'] = (
    df_with_cost['total_revenue'] - df_with_cost['cost_price'] * df_with_cost['quantity']
)

print('Profit by category:')
print(
    df_with_cost.groupby('category')
    .agg(total_revenue=('total_revenue', 'sum'), total_profit=('profit', 'sum'))
    .assign(margin_pct=lambda x: (x['total_profit'] / x['total_revenue'] * 100).round(1))
    .sort_values('total_profit', ascending=False)
)

---
## Section 6 — pd.concat() — Stack Multiple Files

`concat()` stacks DataFrames with the same columns — more rows, not more columns.

Real scenario: you receive monthly CSV exports and must combine them.

In [ ]:
# Create 6 monthly files (Jan–Jun 2024) to simulate the real scenario
df['month'] = df['order_date'].dt.month
df['year']  = df['order_date'].dt.year
data_2024   = df[df['year'] == 2024].copy()

saved = []
for m in sorted(data_2024['month'].dropna().unique()):
    m = int(m)
    chunk = data_2024[data_2024['month'] == m]
    fname = f'sales_2024_{m:02d}.csv'
    chunk.to_csv(fname, index=False)
    saved.append(fname)
    print(f'  {fname}: {len(chunk)} orders')

print(f'\n{len(saved)} files created.')

In [ ]:
# Professional pattern: loop → collect into list → concat once at the end
# Never concat inside a loop — it rebuilds a DataFrame every iteration (very slow)

import glob

all_files = sorted(glob.glob('sales_2024_*.csv'))
print('Files found:', all_files)

dfs = []
for f in all_files:
    temp = pd.read_csv(f)
    temp['source'] = f         # always track which file each row came from
    dfs.append(temp)

all_2024 = pd.concat(dfs, ignore_index=True)  # ignore_index resets row numbers to 0,1,2...
print(f'\nCombined: {all_2024.shape}')

In [ ]:
# Quick clean and analyse the combined file
all_2024['unit_price'] = pd.to_numeric(
    all_2024['unit_price'].astype(str).str.replace('Rs.', '', regex=False).str.strip(),
    errors='coerce'
)
all_2024 = all_2024.dropna(subset=['unit_price'])
all_2024 = all_2024[all_2024['quantity'] > 0]
all_2024['total_revenue'] = all_2024['quantity'] * all_2024['unit_price']

print('Orders per month:')
print(all_2024.groupby('source')['order_id'].count())
print()
print(f'Total 2024 revenue: PKR {all_2024["total_revenue"].sum():,.0f}')

---
## Section 7 — Independent Practice

**10 questions. No code provided. Use only `df` and what you learned in Sections 0–6.**

For each answer:
1. Write working code
2. Print the result clearly
3. Write one sentence interpreting the result in plain English

> These are real analyst tasks. If you can answer all 10, you are ready for a junior data analyst role.

### Q1
Which **product** generates the highest total revenue across all orders?  
Show the top 10 products sorted highest to lowest.  
Also calculate what percentage of total revenue the top product represents.

In [ ]:
# Your code


**Interpretation:** *(write one sentence)*

### Q2
For each **city**, calculate:
- Total revenue
- Total orders
- Cancellation rate (cancelled orders ÷ total orders × 100)
- Average rating

Sort by cancellation rate descending. Which city has the biggest cancellation problem?

In [ ]:
# Hint: calculate cancelled count and total count separately, then divide


**Interpretation:**

### Q3
Find the **month** (by name — January, February, etc.) with the highest number of orders across the entire dataset.  
Show all months sorted by order count.  
Is there a seasonal pattern?

In [ ]:
# Hint: df['order_date'].dt.month_name() gives 'January', 'February', etc.


**Interpretation:**

### Q4
Filter the dataset to **Electronics only**.  
Then find which city buys the most Electronics by revenue.  
Also find the average Electronics order value per city.

In [ ]:
# Hint: filter first (df[df['category'] == 'Electronics']), then groupby


**Interpretation:**

### Q5
Compare **Delivered** vs **Cancelled** orders on these 3 metrics:
- Average order value
- Average quantity per order
- Average rating

Are cancelled orders typically higher or lower value than delivered ones?

In [ ]:
# Hint: filter for Delivered and Cancelled only, then groupby order_status


**Interpretation:**

### Q6
Which **product category** has the highest average rating?  
Which has the lowest?  
Show count of ratings alongside average — categories with very few ratings are less reliable.

In [ ]:
# Hint: use agg() with both 'mean' and 'count' on the rating column
# Note: count() on rating ignores NaN, so it tells you real rated orders


**Interpretation:**

### Q7
Build a table: **city (rows) × order_status (columns) → order count**.  
Add a `cancellation_rate_%` column.  
Which city-status combination has the most orders?

In [ ]:
# Hint: groupby(['city', 'order_status'])['order_id'].count().unstack(fill_value=0)


**Interpretation:**

### Q8
Using `df_full` (the merged DataFrame from Section 5):  
For each **customer tier** (Bronze, Silver, Gold, Platinum):
- Average order value
- Total revenue
- Most purchased category
- Average rating

Do higher-tier customers spend more per order?

In [ ]:
# Hint: most purchased category = lambda x: x.value_counts().index[0]


**Interpretation:**

### Q9
Create two DataFrames:
- `q1_2024` — orders from January, February, March 2024
- `q2_2024` — orders from April, May, June 2024

Stack them with `pd.concat()`. Add a `quarter` column (`'Q1'` or `'Q2'`).  
Then compare total revenue and order count between Q1 and Q2.

In [ ]:
# Hint: df['order_date'].dt.month.isin([1, 2, 3]) for Q1
# Add a 'quarter' column to each before concat


**Interpretation:**

### Q10 — Business Report

You are presenting to the business owner tomorrow.  
Using your results from Q1–Q9, write **5 actionable recommendations** below.  
Each must cite a real number from your analysis.

Format:
> **Recommendation:** [What to do] — because [number-backed reason].

**1.**

**2.**

**3.**

**4.**

**5.**

---
## Quick Reference

| Method | What it does |
|--------|--------------|
| `groupby('col').sum()` | Total per group |
| `groupby('col').mean()` | Average per group |
| `groupby('col').count()` | Number of rows per group |
| `groupby('col').nunique()` | Unique values per group |
| `groupby('col').filter(lambda x: ...)` | Keep groups meeting a condition |
| `groupby('col').agg(new=('src','fn'))` | Multiple stats in one call |
| `groupby(['a','b']).sum().unstack()` | Multi-group → pivot table |
| `pd.concat([s1,s2], axis=1)` | Combine Series as columns |
| `pd.concat([df1,df2], ignore_index=True)` | Stack rows vertically |
| `pd.merge(left, right, on='key', how='left')` | Join tables on shared key |
| `pd.merge(..., left_on='a', right_on='b')` | Join when key has different names |

---
**Tomorrow — Day 2:** Matplotlib + Seaborn — turning GroupBy results into professional charts.